In [ ]:
# ==============================
# Federated Learning on EMNIST Digits (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedProx
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1  # very strict non-IID

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: EMNIST Digits
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.EMNIST(
    root="./data", split="digits", train=True, download=True, transform=transform
)
test_dataset = datasets.EMNIST(
    root="./data", split="digits", train=False, download=True, transform=transform
)

# -------------------------
# Validation Split: 10 fixed clients
# -------------------------
VAL_CLIENTS = list(range(10))  # first 10 clients for validation
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))  # remaining 90 clients for training

# -------------------------
# Dirichlet-based Non-IID Partition
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):
    """
    Returns dict: client_id -> list of indices
    alpha: small -> very strict non-IID
    """
    num_classes = 10
    labels = np.array(dataset.targets)
    idxs = np.arange(len(labels))

    client_dict = {i: [] for i in clients}

    # split indices by class
    class_indices = [idxs[labels == i] for i in range(num_classes)]

    for c_idx in class_indices:
        np.random.shuffle(c_idx)
        proportions = np.random.dirichlet(alpha*np.ones(len(clients)))
        proportions = (np.cumsum(proportions) * len(c_idx)).astype(int)
        proportions = np.concatenate(([0], proportions))
        for i, client in enumerate(clients):
            client_dict[client] += list(c_idx[proportions[i]:proportions[i+1]])

    # Shuffle each client's indices
    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

# Apply Dirichlet partition
client_data = dirichlet_partition(train_dataset, list(range(NUM_CLIENTS)), alpha=ALPHA)

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# -------------------------
# FedProx Regularizer
# -------------------------
def fedprox_loss(model, global_model, mu):
    prox = 0.0
    for w, w0 in zip(model.parameters(), global_model.parameters()):
        prox += torch.sum((w - w0.detach()) ** 2)
    return mu/2 * prox

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_val = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_val += F.cross_entropy(out, y).item() * y.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100*correct/total, loss_val/total

# -------------------------
# Federated Training
# -------------------------
global_model = EMNIST_CNN().to(DEVICE)
scaler = torch.cuda.amp.GradScaler()

# Metrics lists
train_acc, val_acc, test_acc = [], [], []
loss_curve, train_loss_curve, val_loss_curve, test_loss_curve = [], [], [], []
round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):
    start_round = time.time()
    local_weights = []
    local_sizes = []
    local_losses = []

    # select clients randomly for this round
    selected_clients = random.sample(TRAIN_CLIENTS, int(NUM_CLIENTS*CLIENT_FRAC))

    for client in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = optim.SGD(local_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=LOCAL_EPOCHS)

        loader = DataLoader(Subset(train_dataset, client_data[client]), batch_size=BATCH_SIZE, shuffle=True)

        for _ in range(LOCAL_EPOCHS):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    out = local_model(x)
                    loss = F.cross_entropy(out, y)
                    loss += fedprox_loss(local_model, global_model, MU)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

        local_losses.append(loss.item())
        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(client_data[client]))

    # Weighted aggregation
    total = sum(local_sizes)
    new_weights = OrderedDict()
    for k in global_model.state_dict().keys():
        new_weights[k] = sum(
            local_weights[i][k]*(local_sizes[i]/total) for i in range(len(local_weights))
        )
    global_model.load_state_dict(new_weights)

    # Compute metrics
    # Train
    train_loader_all = DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in selected_clients]),
        batch_size=BATCH_SIZE, shuffle=False
    )
    tr_acc, tr_loss = evaluate(global_model, train_loader_all)

    # Validation
    val_loader_all = DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in VAL_CLIENTS]),
        batch_size=BATCH_SIZE, shuffle=False
    )
    va_acc, va_loss = evaluate(global_model, val_loader_all)

    # Test
    te_acc, te_loss = evaluate(global_model, DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False))

    # Save metrics
    loss_curve.append(np.mean(local_losses))
    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)
    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    round_t = time.time() - start_round
    round_time.append(round_t)
    total_time.append(time.time() - start_total)

    print(f"Round {rnd+1:03d} | "
          f"LocalLoss {loss_curve[-1]:.4f} | "
          f"TrainLoss {train_loss_curve[-1]:.4f} | "
          f"ValLoss {val_loss_curve[-1]:.4f} | "
          f"TestLoss {test_loss_curve[-1]:.4f} | "
          f"Train {train_acc[-1]:.4f} | "
          f"Val {val_acc[-1]:.4f} | Test {test_acc[-1]:.4f} | "
          f"Round Time {round_time[-1]:.2f}s | Total Time {total_time[-1]:.2f}s")

Device: cuda


100%|██████████| 562M/562M [00:07<00:00, 79.6MB/s]
/tmp/ipykernel_263/3096718444.py:146: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_263/3096718444.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Round 001 | LocalLoss 0.4829 | TrainLoss 1.7269 | ValLoss 2.0063 | TestLoss 1.9403 | Train 37.9469 | Val 24.2012 | Test 27.2300 | Round Time 202.24s | Total Time 202.24s
Round 002 | LocalLoss 0.1702 | TrainLoss 1.1242 | ValLoss 1.6126 | TestLoss 1.6327 | Train 63.5738 | Val 46.9776 | Test 49.1700 | Round Time 139.72s | Total Time 341.97s
Round 003 | LocalLoss 0.1521 | TrainLoss 1.2759 | ValLoss 1.2939 | TestLoss 1.2161 | Train 58.3421 | Val 56.6104 | Test 58.2325 | Round Time 123.03s | Total Time 464.99s
Round 004 | LocalLoss 0.1369 | TrainLoss 0.7935 | ValLoss 0.6897 | TestLoss 0.8521 | Train 80.6891 | Val 83.8303 | Test 78.6700 | Round Time 102.34s | Total Time 567.34s
Round 005 | LocalLoss 0.0805 | TrainLoss 0.4565 | ValLoss 0.8265 | TestLoss 0.6677 | Train 85.5038 | Val 71.6746 | Test 77.5150 | Round Time 93.85s | Total Time 661.18s


/tmp/ipykernel_263/3096718444.py:184: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Round 006 | LocalLoss 0.0798 | TrainLoss 0.5102 | ValLoss 0.4761 | TestLoss 0.5512 | Train 84.4501 | Val 85.6482 | Test 82.8875 | Round Time 100.26s | Total Time 761.45s
Round 007 | LocalLoss 0.0899 | TrainLoss 0.4085 | ValLoss 0.4726 | TestLoss 0.4443 | Train 86.7052 | Val 84.5942 | Test 85.5450 | Round Time 91.62s | Total Time 853.07s
Round 008 | LocalLoss 0.0540 | TrainLoss 0.2238 | ValLoss 0.2828 | TestLoss 0.2597 | Train 93.0405 | Val 90.8116 | Test 91.7225 | Round Time 84.88s | Total Time 937.95s
Round 009 | LocalLoss 0.0409 | TrainLoss 0.2004 | ValLoss 0.2696 | TestLoss 0.2622 | Train 93.8290 | Val 91.6342 | Test 91.6850 | Round Time 83.19s | Total Time 1021.14s
Round 010 | LocalLoss 0.0384 | TrainLoss 0.2126 | ValLoss 0.2256 | TestLoss 0.2223 | Train 93.2844 | Val 93.1840 | Test 93.0450 | Round Time 84.36s | Total Time 1105.50s
Round 011 | LocalLoss 0.0264 | TrainLoss 0.1928 | ValLoss 0.1926 | TestLoss 0.2278 | Train 93.8209 | Val 94.1205 | Test 92.4525 | Round Time 79.01s | To

In [ ]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("EMNIST_Digits_fedprox_federated_metrics_0.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("EMNIST_Digits_fedprox_federated_metrics_0.csv")
except:
    print("CSV saved locally as EMNIST_Digits_fedprox_federated_metrics_0.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -------------------------
# Save Global Model
# -------------------------
MODEL_PATH = "Baseline_emnist_digits.pth"

torch.save(global_model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

# -------------------------
# Download model to laptop
# -------------------------
from google.colab import files
files.download(MODEL_PATH)

Model saved to: Baseline_emnist.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>